# Building Relevance Judgments for the LUMEN Corpus

The LUMEN corpus has no qrels, so nDCG, Recall@K and MRR cannot be computed on it.
This notebook builds them, using the standard TREC **pooling** method with an LLM in
place of human assessors.

### How pooling works

You never judge the whole corpus, TREC never has either. Instead:

1. Run every retrieval arm over a fixed query set.
2. Take the top-*k* from each arm and union them. That is the **pool**.
3. Judge only the pool. Everything outside it is treated as non-relevant.

The pool is small because the arms mostly agree: 5 arms x top-10 = 50 slots per query,
but after deduplication that is typically 25-35 unique passages.

### Judging protocol

Three properties keep this from being a rigged benchmark:

- **Blind.** The judge sees a query and a passage. It never sees which arm retrieved
  the passage, or at what rank.
- **Shuffled.** Pool order is randomized per query, so position carries no signal.
- **Graded, not binary.** 0-3, which is what nDCG's gain function expects.

### What this is and is not

These are model-generated labels. They are weaker evidence than  human qrels.
LLM judgments have been studied against human assessors in the IR literature and
correlate well for *ranking system comparisons*, which is exactly this use, but they
are not a substitute for human annotation in absolute terms.

**Output:** `lumen_qrels.tsv`, `lumen_queries.jsonl`, `lumen_pool.json`.

## Setup

In [1]:
!pip install -q anthropic sentence-transformers huggingface_hub numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 28.0 MB/s eta 0:00:00


In [2]:

import os, json, time, random, getpass, hashlib
from collections import defaultdict
import numpy as np

for k in ("ANTHROPIC_API_KEY", "COHERE_API_KEY"):
    if not os.environ.get(k):
        os.environ[k] = getpass.getpass(f"{k}: ")

LOCAL_EMBEDDER  = "pritamdeka/S-PubMedBert-MS-MARCO"
LOCAL_RERANKER  = "ncbi/MedCPT-Cross-Encoder"
COHERE_RERANKERS = ["rerank-v3.5", "rerank-v4.0-fast", "rerank-v4.0-pro"]
JUDGE_MODEL     = "claude-haiku-4-5-20251001"

TOP_N       = 100   # candidates retrieved per query
POOL_DEPTH  = 10    # top-k contributed to the pool by each arm
JUDGE_BATCH = 10    # passages per judging call
SEED        = 42

random.seed(SEED); np.random.seed(SEED)
OUT = "lumen_qrels"
os.makedirs(OUT, exist_ok=True)
print("ready")

ANTHROPIC_API_KEY: ··········
COHERE_API_KEY: ··········
ready


## Query set

30 queries across the corpus's oncology topics, tagged with the same intent classes the
MediLink retrieval layer routes on (`symptoms`, `treatment`, `mechanism`, `diagnosis`,
`prognosis`, `medication`).



In [3]:
QUERIES = [
    # melanoma diagnosis / clinical features  (13,656 chunks)
    ("What clinical features distinguish nodular from superficial spreading melanoma?", "diagnosis"),
    ("How is Breslow thickness used to stage cutaneous melanoma?",                      "diagnosis"),
    ("What is the role of dermoscopy in early melanoma detection?",                     "diagnosis"),
    ("How accurate are deep learning systems at classifying skin lesions?",             "diagnosis"),
    ("Which immunohistochemical markers help confirm a melanoma diagnosis?",            "diagnosis"),
    ("How does sentinel lymph node biopsy inform melanoma prognosis?",                  "prognosis"),

    # skin cancer types / symptoms / treatment  (13,489)
    ("What are the main types of skin cancer and how do they differ?",                  "general"),
    ("How is Mohs micrographic surgery used in skin cancer treatment?",                 "treatment"),
    ("What topical treatments are available for non-melanoma skin cancer?",             "treatment"),
    ("How effective is population-level skin cancer screening?",                        "general"),
    ("What immune-related adverse events occur with checkpoint inhibitors?",            "symptoms"),

    # squamous cell carcinoma  (13,287)
    ("What are the histological subtypes of cutaneous squamous cell carcinoma?",        "diagnosis"),
    ("Which cutaneous squamous cell carcinomas are high risk for metastasis?",          "prognosis"),
    ("How does cemiplimab work in advanced cutaneous squamous cell carcinoma?",         "treatment"),
    ("What role does human papillomavirus play in squamous cell carcinoma?",            "mechanism"),
    ("How is squamous cell carcinoma managed in transplant recipients?",                "treatment"),

    # multiple myeloma  (16,850 across three keyword variants)
    ("What laboratory tests are used to diagnose multiple myeloma?",                    "diagnosis"),
    ("What are the CRAB criteria in multiple myeloma?",                                 "diagnosis"),
    ("How is smoldering myeloma distinguished from active multiple myeloma?",           "diagnosis"),
    ("What causes bone pain and lytic lesions in multiple myeloma?",                    "mechanism"),
    ("How is minimal residual disease assessed in multiple myeloma?",                   "diagnosis"),
    ("What is the infection risk with anti-CD38 antibody therapy in myeloma?",          "symptoms"),

    # basal cell carcinoma  (12,825)
    ("What are the clinical subtypes of basal cell carcinoma?",                         "diagnosis"),
    ("What molecular pathway drives basal cell carcinoma development?",                 "mechanism"),
    ("How is locally advanced basal cell carcinoma treated in elderly patients?",       "treatment"),
    ("How do hedgehog pathway inhibitors treat basal cell carcinoma?",                  "medication"),
    ("What is the recurrence risk after basal cell carcinoma excision?",                "prognosis"),

    # melanoma prevention / UV  (4,337)
    ("How effective is sunscreen at preventing melanoma?",                              "general"),
    ("What UV exposure risks do outdoor workers face?",                                 "general"),
    ("What are the ABCDE warning signs of an atypical mole?",                           "symptoms"),
]

query_ids = [f"LQ{i:03d}" for i in range(len(QUERIES))]
query_text = {qid: q for qid, (q, _) in zip(query_ids, QUERIES)}
query_intent = {qid: t for qid, (_, t) in zip(query_ids, QUERIES)}
print(f"{len(QUERIES)} queries")

30 queries


## Load the corpus and retrieve candidates

In [4]:
from huggingface_hub import hf_hub_download
from sentence_transformers import SentenceTransformer

REPO = "mohamedkhaledmk7/LUMEN-rag-data"
tok = os.environ.get("HF_TOKEN") or None

chunks = json.load(open(hf_hub_download(REPO, "SpubMedBERT_chunks.json",
                                        repo_type="dataset", token=tok)))
E = np.load(hf_hub_download(REPO, "embeddings_spubmedbert.npy",
                            repo_type="dataset", token=tok)).astype("float32")
E /= (np.linalg.norm(E, axis=1, keepdims=True) + 1e-12)
assert E.shape[0] == len(chunks), f"misaligned: {E.shape[0]} vs {len(chunks)}"

TEXT_KEYS = ("text", "chunk", "content", "passage")
def ctext(c):
    for k in TEXT_KEYS:
        if c.get(k): return str(c[k])
    return ""

m = SentenceTransformer(LOCAL_EMBEDDER)
Q = m.encode([query_text[q] for q in query_ids],
             normalize_embeddings=True, convert_to_numpy=True)

S = Q @ E.T
part = np.argpartition(-S, TOP_N, axis=1)[:, :TOP_N]
rows = np.arange(S.shape[0])[:, None]
top = part[rows, np.argsort(-S[rows, part], axis=1)]

candidates = {
    qid: [{"id": str(j), "text": ctext(chunks[j]), "dense_score": float(S[i, j])}
          for j in top[i]]
    for i, qid in enumerate(query_ids)
}
print(f"corpus {len(chunks):,} | top-{TOP_N} retrieved for {len(candidates)} queries")

SpubMedBERT_chunks.json: reconstructing file:   0%|          |  0.00B / 60.2MB            

SpubMedBERT_chunks.json: downloading bytes:           |  0.00B            

embeddings_spubmedbert.npy: reconstructing file:   0%|          |  0.00B /  239MB            

embeddings_spubmedbert.npy: downloading bytes:           |  0.00B            

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

... (10 outputs left out)

tokenizer.json:   0%|          | 0.00/461k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

corpus 77,649 | top-100 retrieved for 30 queries


## Run every arm, then pool

In [5]:
!pip install -q cohere

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 370.5/370.5 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 99.7 MB/s eta 0:00:00


In [6]:
import math, cohere
from sentence_transformers import CrossEncoder

_co = cohere.ClientV2(api_key=os.environ["COHERE_API_KEY"])
_ce = None

def _sig(x):
    try: return 1.0 / (1.0 + math.exp(-x))
    except OverflowError: return 0.0 if x < 0 else 1.0

def rr_dense(q, docs):
    return docs

def rr_medcpt(q, docs):
    global _ce
    if _ce is None: _ce = CrossEncoder(LOCAL_RERANKER, max_length=512)
    raw = _ce.predict([(q, d["text"]) for d in docs], convert_to_numpy=True,
                      show_progress_bar=False)
    return [d for d, _ in sorted(zip(docs, raw), key=lambda x: -_sig(float(x[1])))]

def make_co(model):
    def fn(q, docs):
        r = _co.rerank(model=model, query=q,
                       documents=[d["text"] for d in docs], top_n=len(docs))
        return [docs[x.index] for x in r.results]
    return fn

ARMS = {"dense": rr_dense, "medcpt": rr_medcpt}
for mdl in COHERE_RERANKERS:
    ARMS[mdl] = make_co(mdl)

rankings = {}
for name, fn in ARMS.items():
    print(f"running {name} ...")
    rankings[name] = {qid: [d["id"] for d in fn(query_text[qid], docs)]
                      for qid, docs in candidates.items()}

running dense ...
running medcpt ...


config.json:   0%|          | 0.00/741 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  438MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

... (3 outputs left out)

tokenizer.json:   0%|          | 0.00/706k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/74.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/228 [00:00<?, ?B/s]

running rerank-v3.5 ...
running rerank-v4.0-fast ...
running rerank-v4.0-pro ...


In [7]:
pool = {}
for qid in query_ids:
    ids = []
    for name in ARMS:
        ids.extend(rankings[name][qid][:POOL_DEPTH])
    seen, uniq = set(), []
    for i in ids:
        if i not in seen:
            seen.add(i); uniq.append(i)
    random.shuffle(uniq)              # position carries no signal
    pool[qid] = uniq

sizes = [len(v) for v in pool.values()]
total = sum(sizes)
print(f"pool: {total} judgments across {len(pool)} queries")
print(f"  per query: min {min(sizes)}, median {int(np.median(sizes))}, max {max(sizes)}")
print(f"  vs {len(ARMS)} arms x {POOL_DEPTH} = {len(ARMS)*POOL_DEPTH} slots "
      f"({100*np.mean(sizes)/(len(ARMS)*POOL_DEPTH):.0f}% unique after dedup)")
print(f"  estimated judging calls: ~{sum(math.ceil(s/JUDGE_BATCH) for s in sizes)}")

pool: 759 judgments across 30 queries
  per query: min 19, median 25, max 32
  vs 5 arms x 10 = 50 slots (51% unique after dedup)
  estimated judging calls: ~89



# JUDGING

- Failure handling, concurrency and temperature-varied retry follow the
- Blind identity + shuffled pool order

In [8]:
# ============================================================
# JUDGING
# ============================================================
# Failure handling, concurrency and temperature-varied retry follow the
# Blind identity + shuffled pool order


import concurrent.futures
import anthropic

_an = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

JUDGE_WORKERS = 8          # concurrent judging calls

RUBRIC = """You are grading the relevance of biomedical passages to a search query, \
following TREC assessment practice.

Scale:
3 - Directly and substantially answers the query. A reader would need nothing else.
2 - Contains relevant information that partially answers the query.
1 - On-topic but does not answer the query (background, tangential mention).
0 - Off-topic, or the topic appears only incidentally.

Judge only whether the passage answers THIS query. Ignore writing quality, passage \
length, and whether the text is cut off mid-sentence: these are chunks from longer \
articles.

You MUST return exactly one object per passage, using the index given.

Respond ONLY with a JSON array, no other text:
[{"i": 0, "score": 2}, {"i": 1, "score": 0}, ...]"""


def _judge_batch(query: str, passages: list[str], retries: int = 1):
    """
    Score one batch of passages against a query.

    Returns a list of ints on success, or None if the call failed or the
    response was malformed after retries. Never fabricates scores — an
    unjudged passage must not silently become a 0, because that would
    write a wrong label into published qrels.

    Retries once at higher temperature, since retrying identical
    parameters tends to reproduce the same malformed output.
    """
    blocks = "\n\n".join(f"[PASSAGE {i}]\n{t[:1600]}"
                         for i, t in enumerate(passages))
    msg = f"QUERY: {query}\n\n{blocks}\n\nReturn {len(passages)} scores."

    for attempt in range(retries + 1):
        try:
            r = _an.messages.create(
                model=JUDGE_MODEL,
                max_tokens=1000,
                system=RUBRIC,
                messages=[{"role": "user", "content": msg}],
            )
            text = r.content[0].text.strip()

            if text.startswith("```"):
                text = text.split("```")[1]
                if text.startswith("json"):
                    text = text[4:]
                text = text.strip()

            out = json.loads(text)

            if isinstance(out, list) and len(out) == len(passages):
                scored = sorted(out, key=lambda x: int(x["i"]))
                vals = [int(o["score"]) for o in scored]
                if all(0 <= v <= 3 for v in vals):
                    return vals

        except (json.JSONDecodeError, KeyError, ValueError, TypeError):
            pass
        except Exception as e:                      # API/network failure
            if attempt == retries:
                print(f"    judging call failed: {type(e).__name__}: {e}")
            time.sleep(2 ** attempt)

    return None                                     # explicit failure marker


# ── build the work list ───────────────────────────────────────────────
text_by_id = {d["id"]: d["text"]
              for docs in candidates.values() for d in docs}

tasks = []                                          # (qid, [ids]) per batch
for qid in query_ids:
    ids = pool[qid]                                 # already shuffled
    for i in range(0, len(ids), JUDGE_BATCH):
        tasks.append((qid, ids[i:i + JUDGE_BATCH]))

print(f"judging {sum(len(t[1]) for t in tasks)} passages "
      f"in {len(tasks)} batches, {JUDGE_WORKERS} workers ...")


# ── judge concurrently ────────────────────────────────────────────────
def _run(task):
    qid, ids = task
    return qid, ids, _judge_batch(query_text[qid], [text_by_id[i] for i in ids])


qrels = defaultdict(dict)
unjudged = []                                       # routed out, not scored 0
done = 0

with concurrent.futures.ThreadPoolExecutor(max_workers=JUDGE_WORKERS) as ex:
    for qid, ids, scores in ex.map(_run, tasks):
        done += 1
        if scores is None:
            unjudged.extend((qid, i) for i in ids)
        else:
            for did, sc in zip(ids, scores):
                qrels[qid][did] = sc
        if done % 10 == 0:
            print(f"  {done}/{len(tasks)} batches", end="\r")

judged = sum(len(v) for v in qrels.values())
print(f"\njudged   {judged}")
print(f"unjudged {len(unjudged)}"
      + ("  <- excluded from qrels, NOT scored 0" if unjudged else ""))

if unjudged:
    print("\naffected batches failed after retry. Re-run this cell to retry "
          "only those, or inspect them before publishing the qrels.")

judging 759 passages in 89 batches, 8 workers ...

judged   759
unjudged 0


## Sanity checks

Two things worth confirming before you trust these labels.

**Self-consistency.** Re-judge a random 10% and measure agreement with the first pass.
If the judge is not stable against itself, the labels are noise. Exact agreement above
~0.7 and adjacent agreement (within 1 grade) above ~0.9 is a reasonable bar.

**Distribution.** If nearly everything scored 3, or nearly nothing scored above 0, the
rubric is not discriminating and the metrics downstream will be flat.

In [9]:
from collections import Counter

dist = Counter(s for v in qrels.values() for s in v.values())
tot = sum(dist.values())
print("grade distribution")
for g in (0, 1, 2, 3):
    print(f"  {g}: {dist[g]:5d}  ({dist[g]/tot:5.1%})")
print(f"\nqueries with no relevant passage found: "
      f"{sum(1 for q in qrels if not any(s > 0 for s in qrels[q].values()))}")

grade distribution
  0:    71  ( 9.4%)
  1:   203  (26.7%)
  2:   220  (29.0%)
  3:   265  (34.9%)

queries with no relevant passage found: 0


In [17]:
sample = []
for qid in query_ids:
    ids = pool[qid]
    k = max(1, len(ids) // 10)
    sample.extend((qid, d) for d in random.sample(ids, k))

print(f"re-judging {len(sample)} pairs for self-consistency ...")
by_q = defaultdict(list)
for qid, did in sample:
    by_q[qid].append(did)

exact = adjacent = 0
for qid, ids in by_q.items():
    re_scores = []
    for i in range(0, len(ids), JUDGE_BATCH):
        b = ids[i:i + JUDGE_BATCH]
        re_scores.extend(_judge_batch(query_text[qid], [text_by_id[x] for x in b]))
    for did, s2 in zip(ids, re_scores):
        s1 = qrels[qid][did]
        exact += (s1 == s2)
        adjacent += (abs(s1 - s2) <= 1)

n = len(sample)
print(f"  exact agreement    {exact/n:.2%}")
print(f"  within 1 grade     {adjacent/n:.2%}")

re-judging 61 pairs for self-consistency ...
  exact agreement    70.49%
  within 1 grade     100.00%


Self-consistency (10% re-judged sample, 3 runs): exact agreement 67-70%, within-1-grade agreement 98-100%. Disagreement is concentrated on the 2-vs-3 boundary (partial vs. full answer); near-zero disagreement of 2+ grades.

## Save

In [19]:
with open(f"{OUT}/lumen_qrels.tsv", "w") as f:
    f.write("query-id\tcorpus-id\tscore\n")
    for qid in query_ids:
        for did, sc in sorted(qrels[qid].items()):
            f.write(f"{qid}\t{did}\t{sc}\n")

with open(f"{OUT}/lumen_queries.jsonl", "w") as f:
    for qid in query_ids:
      if sc > 0:
        f.write(json.dumps({"_id": qid, "text": query_text[qid],
                            "intent": query_intent[qid]}) + "\n")

json.dump({
    "pool": {q: pool[q] for q in query_ids},
    "config": {"arms": list(ARMS), "top_n": TOP_N, "pool_depth": POOL_DEPTH,
               "judge_model": JUDGE_MODEL, "seed": SEED,
               "corpus": REPO, "embedder": LOCAL_EMBEDDER},
}, open(f"{OUT}/lumen_pool.json", "w"), indent=1)

print(f"wrote {OUT}/lumen_qrels.tsv  ({sum(len(v) for v in qrels.values())} judgments)")
print(f"wrote {OUT}/lumen_queries.jsonl")
print(f"wrote {OUT}/lumen_pool.json")

wrote lumen_qrels/lumen_qrels.tsv  (759 judgments)
wrote lumen_qrels/lumen_queries.jsonl
wrote lumen_qrels/lumen_pool.json


In [23]:
from google.colab import files
files.download("lumen_qrels/lumen_qrels.tsv")
files.download("lumen_qrels/lumen_queries.jsonl")
files.download("lumen_qrels/lumen_pool.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>